# 💰 Task 2: Credit Risk Prediction
**Internship: DevelopersHub Corporation — Data Science & Analytics**

---
## 1. Introduction & Problem Statement
Predict whether a loan applicant is likely to default, using applicant demographic and financial data.

**Dataset:** Loan Prediction Dataset  
**Model:** Logistic Regression

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
sns.set_theme(style='whitegrid')
print('✅ Libraries imported')

## 2. Dataset Understanding

In [ ]:
df = pd.read_csv('loan_prediction.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()
print('\nMissing values:\n', df.isnull().sum())

## 3. Data Cleaning & Preparation

In [ ]:
df.drop(columns=['Loan_ID'], inplace=True)

# Fill categorical missing with mode, numeric with median
for col in ['Gender','Married','Dependents','Self_Employed']:
    df[col].fillna(df[col].mode()[0], inplace=True)
for col in ['LoanAmount','Loan_Amount_Term','Credit_History']:
    df[col].fillna(df[col].median(), inplace=True)

print('Missing after cleaning:', df.isnull().sum().sum())

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1,3, figsize=(15,4))
sns.histplot(df['LoanAmount'], kde=True, ax=axes[0]); axes[0].set_title('Loan Amount Distribution')
sns.countplot(data=df, x='Education', hue='Loan_Status', ax=axes[1]); axes[1].set_title('Education vs Loan Status')
sns.histplot(df['ApplicantIncome'], kde=True, ax=axes[2]); axes[2].set_title('Applicant Income Distribution')
plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150)
plt.show()

## 5. Model Training

In [ ]:
df_enc = df.copy()
le = LabelEncoder()
cat_cols = ['Gender','Married','Dependents','Education','Self_Employed','Property_Area']
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])
df_enc['Loan_Status'] = df_enc['Loan_Status'].map({'Y':1,'N':0})

X = df_enc.drop(columns=['Loan_Status'])
y = df_enc['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
print('✅ Model trained')

## 6. Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.4f}')
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Default','Approved'], cmap='Blues')
plt.title('Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

## 7. Conclusion
- Credit history was the strongest signal for loan approval.
- Model achieves reasonable accuracy on held-out test data.
- Missing values were handled via mode/median imputation rather than dropping rows.